# ViTrueVoice — Pipeline Tiền Xử Lý Dữ Liệu & Sinh Spoof Trên Kaggle GPU

Hệ thống phát hiện giọng nói giả tiếng Việt qua kênh thoại nén.
Notebook thực thi trọn vẹn:
1. Chuẩn hóa âm thanh Bonafide 16kHz mono PCM.
2. Tự động sinh dữ liệu Spoof qua F5-TTS và VieNeu-TTS (Adaptive mode) kèm nén viễn thông G.711 / AMR-NB.
3. Lập bảng manifest 17 cột theo `DataColunm_Plan.docx` và phân vùng Speaker-Disjoint (Train, Dev, Test).
4. Đóng gói kết quả về file ZIP để tải về máy local qua `manage_kaggle.py output`.


In [ ]:
# 1. Cài đặt các thư viện cần thiết trên Kaggle GPU
import subprocess
import sys
import os

print('📦 Đang cài đặt thư viện phụ thuộc (F5-TTS, VieNeu-TTS, SpeechBrain, Codecs)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'f5-tts', 'vieneu', 'speechbrain', 'jiwer', 'soundfile', 'librosa'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libopencore-amrnb-dev'], check=False)

import torch
print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ CẢNH BÁO: Không có GPU! Pipeline sẽ chạy rất chậm.')
print('=' * 60)


In [ ]:
# 2. Khởi tạo package vitruevoice trên Kaggle container
import base64
import zlib
import json
from pathlib import Path

PKG_B64 = '...'

pkg_data = json.loads(zlib.decompress(base64.b64decode(PKG_B64.encode('ascii'))).decode('utf-8'))
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./out')
pkg_root = working_dir / 'vitruevoice'
pkg_root.mkdir(parents=True, exist_ok=True)

for rel_path, content in pkg_data.items():
    file_path = pkg_root / rel_path
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding='utf-8')

if str(working_dir) not in sys.path:
    sys.path.insert(0, str(working_dir))

print(f'✅ Đã khởi tạo thành công package vitruevoice tại: {pkg_root}')


In [ ]:
# 3. Tiền xử lý Bonafide Audio: 16kHz Mono 16-bit PCM & trích xuất Ref 3-5s
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import subprocess
from pathlib import Path

AUDIO_METADATA = json.loads("{\"SPK_0001_01\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"C\u1ea3 hai b\u00ean h\u00e3y c\u1ed1 g\u1eafng b\u00ecnh t\u0129nh l\u1eafng nghe v\u00e0 th\u00f4ng c\u1ea3m cho nhau.\", \"is_ref\": true}, \"SPK_0001_02\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"S\u00e1ng nay th\u1eddi ti\u1ebft \u1edf th\u00e0nh ph\u1ed1 kh\u00e1 m\u00e1t m\u1ebb, b\u1ea7u tr\u1eddi trong xanh v\u00e0 \u00edt n\u1eafng.\", \"is_ref\": true}, \"SPK_0001_03\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"Nh\u1eefng k\u1ef9 n\u0103ng x\u1eed l\u00fd th\u00f4ng tin an to\u00e0n tr\u00ean m\u1ea1ng x\u00e3 h\u1ed9i ng\u00e0y c\u00e0ng quan tr\u1ecdng.\", \"is_ref\": true}, \"SPK_0001_04\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"Em \u0111\u00e3 ki\u1ec3m tra l\u1ea1i to\u00e0n b\u1ed9 l\u1ecbch tr\u00ecnh l\u00e0m vi\u1ec7c c\u1ee7a tu\u1ea7n t\u1edbi r\u1ed3i nh\u00e9.\", \"is_ref\": true}, \"SPK_0001_05\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"M\u1ed7i bu\u1ed5i t\u1ed1i c\u1ea3 gia \u0111\u00ecnh th\u01b0\u1eddng qu\u00e2y qu\u1ea7n b\u00ean m\u00e2m c\u01a1m n\u00f3ng h\u1ed5i.\", \"is_ref\": true}, \"SPK_0001_06\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"M\u1eb9 \u01a1i t\u1ed1i nay con ph\u1ea3i \u1edf l\u1ea1i tr\u01b0\u1eddng l\u00e0m th\u00eam b\u00e0i t\u1eadp nh\u00f3m n\u00ean v\u1ec1 tr\u1ec5 m\u1ed9t ch\u00fat, b\u1ed1 m\u1eb9 c\u1ee9 \u0103n c\u01a1m tr\u01b0\u1edbc \u0111i \u0111\u1eebng ch\u1edd con nh\u00e9.\", \"is_ref\": false}, \"SPK_0001_07\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"Cu\u1ed1i tu\u1ea7n n\u00e0y n\u1ebfu kh\u00f4ng c\u00f3 l\u1ecbch h\u1ecdc b\u00f9 th\u00ec con s\u1ebd b\u1eaft xe bu\u00fdt v\u1ec1 th\u0103m \u00f4ng b\u00e0 ngo\u1ea1i v\u00e0 mua \u00edt tr\u00e1i c\u00e2y t\u01b0\u01a1i v\u1ec1 bi\u1ebfu \u00f4ng b\u00e0.\", \"is_ref\": false}, \"SPK_0001_08\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"B\u1ed1 nh\u1edb u\u1ed1ng thu\u1ed1c huy\u1ebft \u00e1p \u0111\u00fang gi\u1edd nh\u01b0 b\u00e1c s\u0129 \u0111\u00e3 d\u1eb7n trong \u0111\u01a1n nh\u00e9, tu\u1ea7n sau con xin ngh\u1ec9 bu\u1ed5i s\u00e1ng \u0111\u1ec3 \u0111\u01b0a b\u1ed1 \u0111i t\u00e1i kh\u00e1m.\", \"is_ref\": false}, \"SPK_0001_09\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"H\u00f4m qua ch\u1ed7 con \u1edf m\u01b0a l\u1edbn ng\u1eadp h\u1ebft c\u1ea3 con \u0111\u01b0\u1eddng tr\u01b0\u1edbc c\u1ed5ng k\u00fd t\u00fac x\u00e1, may m\u00e0 con \u0111i h\u1ecdc v\u1ec1 s\u1edbm n\u00ean kh\u00f4ng b\u1ecb \u01b0\u1edbt h\u1ebft qu\u1ea7n \u00e1o.\", \"is_ref\": false}, \"SPK_0001_10\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"Em m\u1edbi g\u1eedi file t\u00e0i li\u1ec7u thuy\u1ebft tr\u00ecnh qua Zalo cho anh r\u1ed3i \u0111\u00f3, anh t\u1ea3i v\u1ec1 m\u00e1y t\u00ednh xem th\u1eed b\u1ed1 c\u1ee5c \u0111\u00e3 h\u1ee3p l\u00fd ch\u01b0a nh\u00e9.\", \"is_ref\": false}, \"SPK_0001_11\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"Chi\u1ec1u nay tan ca xong m\u00ecnh gh\u00e9 si\u00eau th\u1ecb g\u1ea7n nh\u00e0 mua th\u00eam \u00edt rau xanh, th\u1ecbt b\u00f2 v\u1edbi s\u1eefa t\u01b0\u01a1i \u0111\u1ec3 chu\u1ea9n b\u1ecb n\u1ea5u b\u1eefa t\u1ed1i lu\u00f4n nha.\", \"is_ref\": false}, \"SPK_0001_12\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"\u0110i\u1ec7n tho\u1ea1i c\u1ee7a con d\u1ea1o n\u00e0y pin h\u01a1i y\u1ebfu n\u00ean th\u1ec9nh tho\u1ea3ng b\u1ecb s\u1eadp ngu\u1ed3n, n\u1ebfu m\u1eb9 g\u1ecdi kh\u00f4ng \u0111\u01b0\u1ee3c th\u00ec c\u1ee9 nh\u1eafn tin v\u00e0o Zalo cho con.\", \"is_ref\": false}, \"SPK_0001_13\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"Alo m\u1eb9 \u01a1i c\u1ee9u con v\u1edbi, con v\u1eeba m\u1edbi va ch\u1ea1m xe m\u00e1y ngo\u00e0i ng\u00e3 t\u01b0, ng\u01b0\u1eddi ta b\u1eaft \u0111\u1ec1n hai m\u01b0\u01a1i tri\u1ec7u \u0111\u1ed3ng, m\u1eb9 chuy\u1ec3n g\u1ea5p gi\u00fap con nh\u00e9!\", \"is_ref\": false}, \"SPK_0001_14\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"B\u1ed1 \u01a1i m\u00e1y con \u0111ang h\u1ecfng loa kh\u00f4ng nghe g\u1ecdi l\u1ea1i \u0111\u01b0\u1ee3c \u0111\u00e2u, b\u1ed1 chuy\u1ec3n kho\u1ea3n ngay m\u01b0\u1eddi l\u0103m tri\u1ec7u v\u00e0o s\u1ed1 t\u00e0i kho\u1ea3n con v\u1eeba nh\u1eafn tr\u00ean Zalo \u0111i!\", \"is_ref\": false}, \"SPK_0001_15\": {\"speaker_id\": \"SPK_0001\", \"transcript\": \"M\u1eb9 \u0111\u1eebng g\u1ecdi l\u1ea1i cho con l\u00fac n\u00e0y v\u00ec con \u0111ang l\u00e0m vi\u1ec7c v\u1edbi c\u00f4ng an giao th\u00f4ng, m\u1eb9 chuy\u1ec3n ti\u1ec1n l\u1eb9 l\u00ean \u0111\u1ec3 ng\u01b0\u1eddi ta cho con l\u1ea5y xe v\u1ec1.\", \"is_ref\": false}, \"SPK_0004_01\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Xin ch\u00e0o qu\u00fd kh\u00e1ch, cu\u1ed9c g\u1ecdi n\u00e0y \u0111\u01b0\u1ee3c ghi \u00e2m \u0111\u1ec3 \u0111\u1ea3m b\u1ea3o ch\u1ea5t l\u01b0\u1ee3ng d\u1ecbch v\u1ee5.\", \"is_ref\": true}, \"SPK_0004_02\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"H\u1ec7 th\u1ed1ng ng\u00e2n h\u00e0ng s\u1ed1 hi\u1ec7n \u0111ang n\u00e2ng c\u1ea5p b\u1ea3o m\u1eadt \u0111\u1ecbnh k\u1ef3 v\u00e0o ban \u0111\u00eam.\", \"is_ref\": true}, \"SPK_0004_03\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"M\u1ecdi giao d\u1ecbch chuy\u1ec3n kho\u1ea3n tr\u1ef1c tuy\u1ebfn \u0111\u1ec1u c\u1ea7n \u0111\u01b0\u1ee3c x\u00e1c th\u1ef1c sinh tr\u1eafc h\u1ecdc.\", \"is_ref\": true}, \"SPK_0004_04\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Qu\u00fd kh\u00e1ch vui l\u00f2ng ki\u1ec3m tra k\u1ef9 th\u00f4ng tin t\u00ean ng\u01b0\u1eddi th\u1ee5 h\u01b0\u1edfng tr\u01b0\u1edbc khi x\u00e1c nh\u1eadn.\", \"is_ref\": true}, \"SPK_0004_05\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Ch\u00fac anh ch\u1ecb c\u00f3 m\u1ed9t ng\u00e0y l\u00e0m vi\u1ec7c hi\u1ec7u qu\u1ea3 v\u00e0 nhi\u1ec1u ni\u1ec1m vui.\", \"is_ref\": true}, \"SPK_0004_06\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Ch\u00e0o anh, em g\u1ecdi t\u1eeb b\u1ed9 ph\u1eadn ch\u0103m s\u00f3c kh\u00e1ch h\u00e0ng \u0111\u1ec3 th\u00f4ng b\u00e1o th\u1ebb t\u00edn d\u1ee5ng c\u1ee7a anh \u0111\u00e3 \u0111\u01b0\u1ee3c gia h\u1ea1n th\u00e0nh c\u00f4ng v\u00e0 s\u1ebd giao v\u1ec1 \u0111\u1ecba ch\u1ec9 nh\u00e0.\", \"is_ref\": false}, \"SPK_0004_07\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"N\u1ebfu anh ch\u1ecb c\u00f3 nhu c\u1ea7u m\u1edf s\u1ed5 ti\u1ebft ki\u1ec7m tr\u1ef1c tuy\u1ebfn v\u1edbi l\u00e3i su\u1ea5t \u01b0u \u0111\u00e3i th\u00ec c\u00f3 th\u1ec3 thao t\u00e1c tr\u1ef1c ti\u1ebfp ngay tr\u00ean \u1ee9ng d\u1ee5ng \u0111i\u1ec7n tho\u1ea1i.\", \"is_ref\": false}, \"SPK_0004_08\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Hi\u1ec7n t\u1ea1i chi nh\u00e1nh ng\u00e2n h\u00e0ng l\u00e0m vi\u1ec7c t\u1eeb th\u1ee9 Hai \u0111\u1ebfn th\u1ee9 S\u00e1u h\u00e0ng tu\u1ea7n, bu\u1ed5i s\u00e1ng b\u1eaft \u0111\u1ea7u t\u1eeb b\u1ea3y gi\u1edd r\u01b0\u1ee1i \u0111\u1ebfn m\u01b0\u1eddi m\u1ed9t gi\u1edd r\u01b0\u1ee1i.\", \"is_ref\": false}, \"SPK_0004_09\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"\u0110\u1ec3 c\u1eadp nh\u1eadt d\u1eef li\u1ec7u sinh tr\u1eafc h\u1ecdc khu\u00f4n m\u1eb7t, qu\u00fd kh\u00e1ch ch\u1ec9 c\u1ea7n chu\u1ea9n b\u1ecb th\u1ebb c\u0103n c\u01b0\u1edbc c\u00f4ng d\u00e2n g\u1eafn chip v\u00e0 \u0111i\u1ec7n tho\u1ea1i c\u00f3 h\u1ed7 tr\u1ee3 k\u1ebft n\u1ed1i NFC.\", \"is_ref\": false}, \"SPK_0004_10\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Em xin ph\u00e9p g\u1eedi b\u1ea3ng k\u00ea chi ti\u1ebft c\u00e1c kho\u1ea3n chi ti\u00eau trong k\u1ef3 sao k\u00ea th\u00e1ng M\u01b0\u1eddi qua \u0111\u1ecba ch\u1ec9 th\u01b0 \u0111i\u1ec7n t\u1eed m\u00e0 ch\u1ecb \u0111\u00e3 \u0111\u0103ng k\u00fd v\u1edbi ng\u00e2n h\u00e0ng.\", \"is_ref\": false}, \"SPK_0004_11\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Khi g\u1eb7p s\u1ef1 c\u1ed1 nu\u1ed1t th\u1ebb t\u1ea1i m\u00e1y r\u00fat ti\u1ec1n t\u1ef1 \u0111\u1ed9ng, anh ch\u1ecb vui l\u00f2ng g\u1ecdi ngay \u0111\u1ebfn \u0111\u01b0\u1eddng d\u00e2y n\u00f3ng in tr\u00ean th\u00e2n m\u00e1y \u0111\u1ec3 \u0111\u01b0\u1ee3c h\u1ed7 tr\u1ee3 kh\u00f3a th\u1ebb.\", \"is_ref\": false}, \"SPK_0004_12\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"C\u1ea3m \u01a1n qu\u00fd kh\u00e1ch \u0111\u00e3 lu\u00f4n tin t\u01b0\u1edfng v\u00e0 \u0111\u1ed3ng h\u00e0nh c\u00f9ng c\u00e1c d\u1ecbch v\u1ee5 t\u00e0i ch\u00ednh c\u1ee7a ch\u00fang t\u00f4i trong su\u1ed1t th\u1eddi gian v\u1eeba qua.\", \"is_ref\": false}, \"SPK_0004_13\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Th\u00f4ng b\u00e1o kh\u1ea9n: t\u00e0i kho\u1ea3n ng\u00e2n h\u00e0ng c\u1ee7a anh v\u1eeba ph\u00e1t sinh giao d\u1ecbch b\u1ea5t th\u01b0\u1eddng tr\u1ecb gi\u00e1 n\u0103m m\u01b0\u01a1i tri\u1ec7u \u0111\u1ed3ng, vui l\u00f2ng \u0111\u1ecdc m\u00e3 OTP \u0111\u1ec3 h\u1ee7y l\u1ec7nh ngay!\", \"is_ref\": false}, \"SPK_0004_14\": {\"speaker_id\": \"SPK_0004\", \"transcript\": \"Ch\u1ecb c\u00f2n kho\u1ea3n n\u1ee3 qu\u00e1 h\u1ea1n th\u1ebb t\u00edn d\u1ee5ng m\u01b0\u1eddi t\u00e1m tri\u1ec7u \u0111\u1ed3ng, n\u1ebfu kh\u00f4ng chuy\u1ec3n kho\u1ea3n thanh to\u00e1n trong ba m\u01b0\u01a1i ph\u00fat n\u1eefa h\u1ed3 s\u01a1 s\u1ebd chuy\u1ec3n sang c\u01a1 quan \u0111i\u1ec1u tra.\", \"is_ref\": false}, \"SPK_0005_01\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"C\u00f4ng d\u00e2n c\u00f3 quy\u1ec1n y\u00eau c\u1ea7u b\u1ea3o v\u1ec7 th\u00f4ng tin d\u1eef li\u1ec7u c\u00e1 nh\u00e2n theo lu\u1eadt \u0111\u1ecbnh.\", \"is_ref\": true}, \"SPK_0005_02\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"M\u1ecdi th\u1ee7 t\u1ee5c h\u00e0nh ch\u00ednh c\u00f4ng hi\u1ec7n nay \u0111\u1ec1u \u0111\u01b0\u1ee3c ni\u00eam y\u1ebft c\u00f4ng khai t\u1ea1i tr\u1ee5 s\u1edf.\", \"is_ref\": true}, \"SPK_0005_03\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"Vi\u1ec7c \u1ee9ng d\u1ee5ng tr\u00ed tu\u1ec7 nh\u00e2n t\u1ea1o gi\u00fap n\u00e2ng cao hi\u1ec7u qu\u1ea3 gi\u1ea3i quy\u1ebft c\u00f4ng vi\u1ec7c.\", \"is_ref\": true}, \"SPK_0005_04\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"\u0110\u1ec1 ngh\u1ecb m\u1ecdi ng\u01b0\u1eddi gi\u1eef tr\u1eadt t\u1ef1 v\u00e0 x\u1ebfp h\u00e0ng theo \u0111\u00fang s\u1ed1 th\u1ee9 t\u1ef1 \u0111\u00e3 ph\u00e1t.\", \"is_ref\": true}, \"SPK_0005_05\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"B\u1ea3n tin th\u1eddi s\u1ef1 bu\u1ed5i t\u1ed1i s\u1ebd \u0111\u01b0\u1ee3c ph\u00e1t s\u00f3ng tr\u1ef1c ti\u1ebfp v\u00e0o l\u00fac m\u01b0\u1eddi ch\u00edn gi\u1edd.\", \"is_ref\": true}, \"SPK_0005_06\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"Ch\u00e0o b\u00e1c, h\u1ed3 s\u01a1 \u0111\u0103ng k\u00fd c\u1ea5p l\u1ea1i gi\u1ea5y ph\u00e9p l\u00e1i xe c\u1ee7a b\u00e1c \u0111\u00e3 \u0111\u01b0\u1ee3c ti\u1ebfp nh\u1eadn tr\u00ean c\u1ed5ng d\u1ecbch v\u1ee5 c\u00f4ng, h\u1eb9n b\u00e1c tu\u1ea7n sau \u0111\u1ebfn nh\u1eadn k\u1ebft qu\u1ea3 nh\u00e9.\", \"is_ref\": false}, \"SPK_0005_07\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"S\u00e1ng th\u1ee9 B\u1ea3y tu\u1ea7n n\u00e0y \u1ee7y ban nh\u00e2n d\u00e2n ph\u01b0\u1eddng c\u00f3 t\u1ed5 ch\u1ee9c bu\u1ed5i h\u01b0\u1edbng d\u1eabn ng\u01b0\u1eddi d\u00e2n c\u00e0i \u0111\u1eb7t \u1ee9ng d\u1ee5ng \u0111\u1ecbnh danh \u0111i\u1ec7n t\u1eed t\u1ea1i h\u1ed9i tr\u01b0\u1eddng khu ph\u1ed1.\", \"is_ref\": false}, \"SPK_0005_08\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"C\u01a1 quan c\u00f4ng an kh\u00f4ng bao gi\u1edd l\u00e0m vi\u1ec7c qua \u0111i\u1ec7n tho\u1ea1i hay y\u00eau c\u1ea7u ng\u01b0\u1eddi d\u00e2n chuy\u1ec3n ti\u1ec1n v\u00e0o t\u00e0i kho\u1ea3n c\u00e1 nh\u00e2n \u0111\u1ec3 ph\u1ee5c v\u1ee5 \u0111i\u1ec1u tra.\", \"is_ref\": false}, \"SPK_0005_09\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"Khi nh\u1eadn \u0111\u01b0\u1ee3c c\u00e1c cu\u1ed9c g\u1ecdi t\u1ef1 x\u01b0ng l\u00e0 vi\u1ec7n ki\u1ec3m s\u00e1t ho\u1eb7c t\u00f2a \u00e1n \u0111e d\u1ecda, b\u00e0 con c\u1ea7n h\u1ebft s\u1ee9c b\u00ecnh t\u0129nh v\u00e0 b\u00e1o ngay cho c\u00f4ng an ph\u01b0\u1eddng g\u1ea7n nh\u1ea5t.\", \"is_ref\": false}, \"SPK_0005_10\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"Anh ch\u1ecb vui l\u00f2ng ki\u1ec3m tra l\u1ea1i th\u00f4ng tin c\u01b0 tr\u00fa c\u1ee7a c\u00e1c th\u00e0nh vi\u00ean trong h\u1ed9 gia \u0111\u00ecnh tr\u00ean ph\u1ea7n m\u1ec1m \u0111\u1ec3 k\u1ecbp th\u1eddi \u0111i\u1ec1u ch\u1ec9nh n\u1ebfu c\u00f3 sai s\u00f3t.\", \"is_ref\": false}, \"SPK_0005_11\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"Bu\u1ed5i tuy\u00ean truy\u1ec1n ph\u1ed5 bi\u1ebfn ph\u00e1p lu\u1eadt v\u1ec1 an to\u00e0n th\u00f4ng tin m\u1ea1ng cho h\u1ecdc sinh v\u00e0 ph\u1ee5 huynh s\u1ebd di\u1ec5n ra v\u00e0o l\u00fac t\u00e1m gi\u1edd s\u00e1ng ng\u00e0y mai.\", \"is_ref\": false}, \"SPK_0005_12\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"N\u1ebfu b\u00e1c c\u1ea7n x\u00e1c nh\u1eadn gi\u1ea5y t\u1edd h\u1ed9 t\u1ecbch th\u00ec ch\u1ec9 c\u1ea7n mang theo b\u1ea3n ch\u00ednh c\u0103n c\u01b0\u1edbc c\u00f4ng d\u00e2n ra b\u1ed9 ph\u1eadn m\u1ed9t c\u1eeda c\u1ee7a ph\u01b0\u1eddng trong gi\u1edd h\u00e0nh ch\u00ednh.\", \"is_ref\": false}, \"SPK_0005_13\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"T\u00f4i g\u1ecdi t\u1eeb ph\u00f2ng c\u1ea3nh s\u00e1t \u0111i\u1ec1u tra, s\u1ed1 \u0111i\u1ec7n tho\u1ea1i v\u00e0 t\u00e0i kho\u1ea3n c\u1ee7a \u00f4ng c\u00f3 li\u00ean quan \u0111\u1ebfn \u0111\u01b0\u1eddng d\u00e2y r\u1eeda ti\u1ec1n xuy\u00ean qu\u1ed1c gia, y\u00eau c\u1ea7u h\u1ee3p t\u00e1c ngay!\", \"is_ref\": false}, \"SPK_0005_14\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"T\u00e0i kho\u1ea3n \u0111\u1ecbnh danh m\u1ee9c \u0111\u1ed9 hai c\u1ee7a b\u00e0 b\u1ecb l\u1ed7i \u0111\u1ed3ng b\u1ed9, b\u00e0 ph\u1ea3i t\u1ea3i ngay ph\u1ea7n m\u1ec1m theo \u0111\u01b0\u1eddng d\u1eabn tr\u00ean Zalo v\u00e0 qu\u00e9t khu\u00f4n m\u1eb7t \u0111\u1ec3 kh\u00f4ng b\u1ecb kh\u00f3a h\u1ed9 kh\u1ea9u.\", \"is_ref\": false}, \"SPK_0005_15\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"H\u1ed3 s\u01a1 \u0111i\u1ec1u tra \u0111ang thu\u1ed9c di\u1ec7n m\u1eadt, y\u00eau c\u1ea7u ch\u1ecb v\u00e0o ph\u00f2ng k\u00edn m\u1ed9t m\u00ecnh, kh\u00f4ng \u0111\u01b0\u1ee3c n\u00f3i cho ng\u01b0\u1eddi nh\u00e0 bi\u1ebft v\u00e0 chuy\u1ec3n to\u00e0n b\u1ed9 ti\u1ec1n sang t\u00e0i kho\u1ea3n t\u1ea1m gi\u1eef.\", \"is_ref\": false}, \"SPK_0005_16\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"Ph\u01b0\u01a1ng ti\u1ec7n mang bi\u1ec3n s\u1ed1 c\u1ee7a anh v\u1eeba g\u00e2y tai n\u1ea1n b\u1ecf ch\u1ea1y t\u1ea1i \u0110\u00e0 N\u1eb5ng, anh ph\u1ea3i n\u1ed9p ngay m\u01b0\u1eddi l\u0103m tri\u1ec7u \u0111\u1ed3ng ti\u1ec1n ph\u1ea1t ngu\u1ed9i \u0111\u1ec3 d\u1eebng l\u1ec7nh truy t\u00ecm.\", \"is_ref\": false}, \"SPK_0005_17\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"Vi\u1ec7n ki\u1ec3m s\u00e1t \u0111\u00e3 k\u00fd l\u1ec7nh phong t\u1ecfa t\u00e0i s\u1ea3n c\u1ee7a gia \u0111\u00ecnh \u00f4ng, mu\u1ed1n ch\u1ee9ng minh ngu\u1ed3n ti\u1ec1n h\u1ee3p ph\u00e1p th\u00ec \u00f4ng ph\u1ea3i chuy\u1ec3n kho\u1ea3n th\u1ea9m tra trong m\u1ed9t gi\u1edd t\u1edbi.\", \"is_ref\": false}, \"SPK_0005_18\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"Ch\u1ecb h\u00e3y \u0111\u1ecdc ngay m\u00e3 s\u1ed1 b\u1ea3o m\u1eadt g\u1eedi v\u1ec1 m\u00e1y \u0111i\u1ec7n tho\u1ea1i \u0111\u1ec3 c\u00e1n b\u1ed9 k\u1ef9 thu\u1eadt k\u00edch ho\u1ea1t l\u1ea1i sim ch\u00ednh ch\u1ee7, n\u1ebfu kh\u00f4ng sim s\u1ebd b\u1ecb thu h\u1ed3i sau m\u01b0\u1eddi l\u0103m ph\u00fat.\", \"is_ref\": false}, \"SPK_0005_19\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"C\u1ee5c thu\u1ebf th\u00f4ng b\u00e1o h\u1ed9 kinh doanh c\u1ee7a anh c\u00f2n n\u1ee3 hai m\u01b0\u01a1i t\u01b0 tri\u1ec7u ti\u1ec1n thu\u1ebf truy thu, y\u00eau c\u1ea7u chuy\u1ec3n kho\u1ea3n n\u1ed9p ngay v\u00e0o t\u00e0i kho\u1ea3n kho b\u1ea1c t\u1ea1m th\u1eddi n\u00e0y.\", \"is_ref\": false}, \"SPK_0005_20\": {\"speaker_id\": \"SPK_0005\", \"transcript\": \"N\u1ebfu b\u00e0 t\u1ef1 \u00fd c\u00fap m\u00e1y ho\u1eb7c b\u00e1o cho ng\u01b0\u1eddi th\u1ee9 ba bi\u1ebft cu\u1ed9c trao \u0111\u1ed5i n\u00e0y th\u00ec ch\u00fang t\u00f4i s\u1ebd l\u1eadp t\u1ee9c c\u1eed l\u1ef1c l\u01b0\u1ee3ng xu\u1ed1ng t\u1eadn nh\u00e0 \u0111\u1ecdc l\u1ec7nh b\u1eaft t\u1ea1m giam.\", \"is_ref\": false}, \"SPK_0006_01\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"D\u1ecbp T\u1ebft Nguy\u00ean \u0111\u00e1n n\u0103m nay c\u1ea3 \u0111\u1ea1i gia \u0111\u00ecnh m\u00ecnh s\u1ebd v\u1ec1 qu\u00ea sum h\u1ecdp.\", \"is_ref\": true}, \"SPK_0006_02\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"Ti\u1ebfng \u0111\u00e0n b\u1ea7u ng\u00e2n nga g\u1ee3i nh\u1edb bi\u1ebft bao k\u1ef7 ni\u1ec7m \u00eam \u0111\u1ec1m th\u1eddi th\u01a1 \u1ea5u.\", \"is_ref\": true}, \"SPK_0006_04\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"Ch\u1ecb nh\u1edb kh\u00f3a k\u1ef9 c\u1eeda c\u1ed5ng v\u00e0 t\u1eaft h\u1ebft thi\u1ebft b\u1ecb \u0111i\u1ec7n tr\u01b0\u1edbc khi \u0111i ng\u1ee7.\", \"is_ref\": true}, \"SPK_0006_05\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"M\u00f9a thu H\u00e0 N\u1ed9i lu\u00f4n mang m\u1ed9t v\u1ebb \u0111\u1eb9p d\u1ecbu d\u00e0ng v\u1edbi h\u01b0\u01a1ng hoa s\u1eefa n\u1ed3ng n\u00e0n.\", \"is_ref\": true}, \"SPK_0006_06\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"D\u00ec Ba \u01a1i ch\u1ee7 nh\u1eadt tu\u1ea7n t\u1edbi l\u00e0 \u0111\u00e1m gi\u1ed7 \u00f4ng n\u1ed9i d\u01b0\u1edbi qu\u00ea r\u1ed3i, gia \u0111\u00ecnh d\u00ec c\u00f3 s\u1eafp x\u1ebfp c\u00f4ng vi\u1ec7c \u0111\u1ec3 v\u1ec1 t\u1eeb chi\u1ec1u th\u1ee9 B\u1ea3y c\u00f9ng t\u1ee5i con kh\u00f4ng \u1ea1?\", \"is_ref\": false}, \"SPK_0006_09\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"B\u00e9 Bi nh\u00e0 em n\u0103m nay v\u00e0o l\u1edbp m\u1ed9t r\u1ed3i n\u00ean t\u1ed1i n\u00e0o hai m\u1eb9 con c\u0169ng ng\u1ed3i t\u1eadp \u0111\u00e1nh v\u1ea7n v\u00e0 r\u00e8n ch\u1eef vi\u1ebft \u0111\u1ebfn t\u1eadn ch\u00edn gi\u1edd t\u1ed1i m\u1edbi xong.\", \"is_ref\": false}, \"SPK_0006_10\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"Em m\u1edbi \u0111\u1ed5i sang d\u00f9ng s\u1ed1 \u0111i\u1ec7n tho\u1ea1i m\u1edbi n\u00e0y \u0111\u1ec3 li\u00ean l\u1ea1c c\u00f4ng vi\u1ec7c, anh Ba l\u01b0u l\u1ea1i s\u1ed1 c\u1ee7a em v\u00e0o danh b\u1ea1 \u0111\u1ec3 khi c\u1ea7n g\u1ecdi cho d\u1ec5 nha.\", \"is_ref\": false}, \"SPK_0006_11\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"\u0110\u1ee3t ngh\u1ec9 l\u1ec5 s\u1eafp t\u1edbi n\u1ebfu nh\u00e0 m\u00ecnh c\u00f3 \u0111i \u0110\u00e0 L\u1ea1t th\u00ec nh\u1edb mang theo \u00e1o kho\u00e1c \u1ea5m cho m\u1ea5y \u0111\u1ee9a nh\u1ecf v\u00ec ban \u0111\u00eam tr\u00ean \u0111\u00f3 tr\u1eddi se l\u1ea1nh l\u1eafm.\", \"is_ref\": false}, \"SPK_0006_18\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"M\u1ea1ng 4G \u1edf ch\u1ed7 em ch\u1eadp ch\u1eddn l\u1eafm kh\u00f4ng nghe cu\u1ed9c g\u1ecdi th\u01b0\u1eddng \u0111\u01b0\u1ee3c \u0111\u00e2u ch\u1ecb \u01a1i, ch\u1ecb \u0111\u1ecdc tin nh\u1eafn tho\u1ea1i n\u00e0y r\u1ed3i chuy\u1ec3n kho\u1ea3n li\u1ec1n hai m\u01b0\u01a1i tri\u1ec7u gi\u00fap em \u0111i!\", \"is_ref\": false}, \"SPK_0006_19\": {\"speaker_id\": \"SPK_0006\", \"transcript\": \"Em v\u1eeba g\u1eedi \u0111\u01b0\u1eddng link b\u00ecnh ch\u1ecdn cu\u1ed9c thi \u1ea3nh c\u1ee7a b\u00e9 nh\u00e0 em \u0111\u00f3, d\u00ec b\u1ea5m v\u00e0o \u0111\u0103ng nh\u1eadp t\u00e0i kho\u1ea3n Zalo r\u1ed3i \u0111\u1ecdc m\u00e3 x\u00e1c nh\u1eadn s\u00e1u s\u1ed1 g\u1eedi l\u1ea1i cho em nha.\", \"is_ref\": false}, \"SPK_0007_01\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"T\u1eadp th\u1ec3 l\u1edbp ch\u00fang ta \u0111\u00e3 ho\u00e0n th\u00e0nh xu\u1ea5t s\u1eafc k\u1ebf ho\u1ea1ch h\u1ecdc t\u1eadp h\u1ecdc k\u1ef3 m\u1ed9t.\", \"is_ref\": true}, \"SPK_0007_02\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"C\u00e1c ph\u01b0\u01a1ng ph\u00e1p n\u00e9n \u00e2m thanh hi\u1ec7n \u0111\u1ea1i gi\u00fap ti\u1ebft ki\u1ec7m b\u0103ng th\u00f4ng truy\u1ec1n d\u1eabn.\", \"is_ref\": true}, \"SPK_0007_03\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"S\u1ef1 ph\u1ed1i h\u1ee3p nh\u1ecbp nh\u00e0ng gi\u1eefa c\u00e1c ph\u00f2ng ban t\u1ea1o n\u00ean th\u00e0nh c\u00f4ng chung.\", \"is_ref\": true}, \"SPK_0007_04\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Th\u1ea7y khuy\u1ebfn kh\u00edch c\u00e1c em ch\u1ee7 \u0111\u1ed9ng t\u00ecm \u0111\u1ecdc th\u00eam c\u00e1c b\u00e0i b\u00e1o khoa h\u1ecdc qu\u1ed1c t\u1ebf.\", \"is_ref\": true}, \"SPK_0007_05\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Bu\u1ed5i l\u1ec5 t\u1ed5ng k\u1ebft v\u00e0 trao gi\u1ea3i th\u01b0\u1edfng s\u1ebd \u0111\u01b0\u1ee3c t\u1ed5 ch\u1ee9c t\u1ea1i h\u1ed9i tr\u01b0\u1eddng l\u1edbn.\", \"is_ref\": true}, \"SPK_0007_06\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Ch\u00e0o anh ch\u1ecb ph\u1ee5 huynh, chi\u1ec1u th\u1ee9 B\u1ea3y tu\u1ea7n n\u00e0y nh\u00e0 tr\u01b0\u1eddng t\u1ed5 ch\u1ee9c h\u1ecdp ph\u1ee5 huynh \u0111\u1ea7u n\u0103m h\u1ecdc v\u00e0o l\u00fac m\u01b0\u1eddi b\u1ed1n gi\u1edd t\u1ea1i ph\u00f2ng h\u1ecdc c\u1ee7a l\u1edbp.\", \"is_ref\": false}, \"SPK_0007_07\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Em t\u1ed5ng h\u1ee3p l\u1ea1i to\u00e0n b\u1ed9 ch\u1ee9ng t\u1eeb thu chi c\u1ee7a d\u1ef1 \u00e1n qu\u00fd ba r\u1ed3i g\u1eedi l\u00ean b\u00e0n l\u00e0m vi\u1ec7c \u0111\u1ec3 chi\u1ec1u nay anh ki\u1ec3m tra v\u00e0 k\u00fd duy\u1ec7t nh\u00e9.\", \"is_ref\": false}, \"SPK_0007_08\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"C\u00e1c b\u1ea1n sinh vi\u00ean l\u01b0u \u00fd h\u1ea1n ch\u00f3t n\u1ed9p b\u00e1o c\u00e1o ti\u1ebfn \u0111\u1ed9 tu\u1ea7n b\u1ed1n l\u00e0 tr\u01b0\u1edbc m\u01b0\u1eddi b\u1ea3y gi\u1edd ng\u00e0y th\u1ee9 S\u00e1u tr\u00ean h\u1ec7 th\u1ed1ng qu\u1ea3n l\u00fd h\u1ecdc t\u1eadp.\", \"is_ref\": false}, \"SPK_0007_09\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Tu\u1ea7n t\u1edbi c\u00f4ng ty m\u00ecnh s\u1ebd \u0111\u00f3n \u0111o\u00e0n kh\u00e1ch h\u00e0ng t\u1eeb Nh\u1eadt B\u1ea3n sang th\u0103m v\u0103n ph\u00f2ng, b\u1ed9 ph\u1eadn h\u00e0nh ch\u00ednh chu\u1ea9n b\u1ecb chu \u0111\u00e1o ph\u00f2ng h\u1ecdp v\u00e0 t\u00e0i li\u1ec7u nh\u00e9.\", \"is_ref\": false}, \"SPK_0007_10\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"C\u00f4 g\u1eedi danh s\u00e1ch c\u00e1c em h\u1ecdc sinh \u0111\u0103ng k\u00fd tham gia chuy\u1ebfn d\u00e3 ngo\u1ea1i ngo\u1ea1i kh\u00f3a v\u00e0o nh\u00f3m l\u1edbp, nh\u1edd c\u00e1c b\u1eadc ph\u1ee5 huynh ki\u1ec3m tra l\u1ea1i th\u00f4ng tin gi\u00fap c\u00f4.\", \"is_ref\": false}, \"SPK_0007_11\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"M\u1ecdi kho\u1ea3n thu h\u1ecdc ph\u00ed v\u00e0 ti\u1ec1n b\u1ea3o hi\u1ec3m y t\u1ebf c\u1ee7a nh\u00e0 tr\u01b0\u1eddng \u0111\u1ec1u \u0111\u01b0\u1ee3c th\u00f4ng b\u00e1o b\u1eb1ng v\u0103n b\u1ea3n ch\u00ednh th\u1ee9c v\u00e0 n\u1ed9p qua t\u00e0i kho\u1ea3n ng\u00e2n h\u00e0ng c\u1ee7a tr\u01b0\u1eddng.\", \"is_ref\": false}, \"SPK_0007_12\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Anh \u0111ang \u0111i c\u00f4ng t\u00e1c \u1edf ngo\u00e0i H\u00e0 N\u1ed9i \u0111\u1ebfn h\u1ebft ng\u00e0y th\u1ee9 N\u0103m m\u1edbi bay v\u1ec1 S\u00e0i G\u00f2n, c\u00f3 c\u00f4ng vi\u1ec7c g\u00ec c\u1ea7n x\u1eed l\u00fd g\u1ea5p th\u00ec em c\u1ee9 nh\u1eafn qua Zalo cho anh.\", \"is_ref\": false}, \"SPK_0007_13\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Alo ch\u1ecb ph\u1ee5 huynh ch\u00e1u Nam ph\u1ea3i kh\u00f4ng, ch\u00e1u v\u1eeba b\u1ecb ng\u00e3 c\u1ea7u thang \u1edf tr\u01b0\u1eddng ch\u1ea5n th\u01b0\u01a1ng v\u00f9ng \u0111\u1ea7u \u0111ang c\u1ea5p c\u1ee9u, ch\u1ecb chuy\u1ec3n g\u1ea5p m\u01b0\u1eddi l\u0103m tri\u1ec7u t\u1ea1m \u1ee9ng m\u1ed5 ngay!\", \"is_ref\": false}, \"SPK_0007_14\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Em k\u1ebf to\u00e1n nghe r\u00f5 \u0111\u00e2y, anh \u0111ang ti\u1ebfp \u0111o\u00e0n thanh tra \u0111\u1ed9t xu\u1ea5t kh\u00f4ng ti\u1ec7n ra ngo\u00e0i, em l\u1ea5y qu\u1ef9 c\u00f4ng ty chuy\u1ec3n g\u1ea5p n\u0103m m\u01b0\u01a1i tri\u1ec7u v\u00e0o s\u1ed1 t\u00e0i kho\u1ea3n n\u00e0y cho anh!\", \"is_ref\": false}, \"SPK_0007_15\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Ch\u00e0o b\u00e1c s\u0129 \u0111ang chu\u1ea9n b\u1ecb \u0111\u01b0a ch\u00e1u v\u00e0o ph\u00f2ng ph\u1eabu thu\u1eadt r\u1ed3i n\u00ean ph\u1ee5 huynh kh\u00f4ng k\u1ecbp ch\u1ea1y l\u00ean vi\u1ec7n \u0111\u00e2u, ph\u1ea3i chuy\u1ec3n kho\u1ea3n vi\u1ec7n ph\u00ed ngay trong ba ph\u00fat n\u1eefa!\", \"is_ref\": false}, \"SPK_0007_16\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"S\u1ebfp \u0111ang h\u1ecdp k\u00edn v\u1edbi \u0111\u1ed1i t\u00e1c chi\u1ebfn l\u01b0\u1ee3c n\u00ean t\u1eaft chu\u00f4ng \u0111i\u1ec7n tho\u1ea1i, em \u1ee9ng tr\u01b0\u1edbc ti\u1ec1n c\u00e1 nh\u00e2n chuy\u1ec3n hai m\u01b0\u01a1i tri\u1ec7u thanh to\u00e1n h\u00f3a \u0111\u01a1n ti\u1ebfp kh\u00e1ch li\u1ec1n cho s\u1ebfp nh\u00e9.\", \"is_ref\": false}, \"SPK_0007_17\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"C\u00f4 gi\u00e1o ch\u1ee7 nhi\u1ec7m th\u00f4ng b\u00e1o \u0111\u1ee3t h\u1ecdc b\u1ed5ng n\u00e0y ch\u1ec9 c\u00f2n duy nh\u1ea5t m\u1ed9t su\u1ea5t cu\u1ed1i c\u00f9ng, ph\u1ee5 huynh chuy\u1ec3n kho\u1ea3n ngay n\u0103m tri\u1ec7u ti\u1ec1n l\u1ec7 ph\u00ed gi\u1eef ch\u1ed7 cho ch\u00e1u nh\u00e9.\", \"is_ref\": false}, \"SPK_0007_18\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Em \u0111\u1eebng g\u1ecdi \u0111i\u1ec7n x\u00e1c minh l\u1ea1i l\u00e0m g\u00ec m\u1ea5t th\u1eddi gian, \u0111\u00e2y l\u00e0 l\u1ec7nh ch\u1ec9 \u0111\u1ea1o tr\u1ef1c ti\u1ebfp t\u1eeb gi\u00e1m \u0111\u1ed1c, c\u1ee9 chuy\u1ec3n kho\u1ea3n xong chi\u1ec1u v\u1ec1 v\u0103n ph\u00f2ng anh k\u00fd phi\u1ebfu chi sau.\", \"is_ref\": false}, \"SPK_0007_19\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"Nh\u00e0 tr\u01b0\u1eddng \u0111ang ho\u00e0n tr\u1ea3 ti\u1ec1n h\u1ecdc ph\u00ed th\u1eeba qua c\u1ed5ng thanh to\u00e1n tr\u1ef1c tuy\u1ebfn, ph\u1ee5 huynh \u0111\u1ecdc ngay m\u00e3 x\u00e1c th\u1ef1c OTP tr\u00ean m\u00e1y \u0111\u1ec3 nh\u1eadn l\u1ea1i b\u1ed1n tri\u1ec7u \u0111\u1ed3ng nh\u00e9.\", \"is_ref\": false}, \"SPK_0007_20\": {\"speaker_id\": \"SPK_0007\", \"transcript\": \"T\u00ecnh tr\u1ea1ng c\u1ee7a ch\u00e1u b\u00e9 \u0111ang r\u1ea5t nguy k\u1ecbch kh\u00f4ng th\u1ec3 ch\u1edd \u0111\u1ee3i th\u00eam \u0111\u01b0\u1ee3c n\u1eefa, gia \u0111\u00ecnh chuy\u1ec3n ngay ba m\u01b0\u01a1i tri\u1ec7u v\u00e0o t\u00e0i kho\u1ea3n c\u00e1 nh\u00e2n c\u1ee7a tr\u01b0\u1edfng khoa c\u1ea5p c\u1ee9u \u0111i!\", \"is_ref\": false}}")

input_roots = [
    Path('/kaggle/input/vitruevoice-raw-bonafide'),
    Path('/kaggle/input'),
    Path('./data'),
]
raw_src_dir = None
for candidate in input_roots:
    if candidate.exists():
        found = list(candidate.glob('**/SPK_0001'))
        if found:
            raw_src_dir = found[0].parent
            break

if not raw_src_dir:
    raise RuntimeError('❌ Không tìm thấy thư mục SPK_* trong /kaggle/input!')

print(f'📂 Nguồn âm thanh Bonafide tìm thấy tại: {raw_src_dir}')
data_root = working_dir / 'data'
raw_out_dir = data_root / 'raw'
ref_out_dir = data_root / 'bonafide_ref'
manifests_dir = data_root / 'manifests'

raw_out_dir.mkdir(parents=True, exist_ok=True)
ref_out_dir.mkdir(parents=True, exist_ok=True)
manifests_dir.mkdir(parents=True, exist_ok=True)

SPEAKER_SPLIT = {
    'SPK_0001': 'train', 'SPK_0004': 'train', 'SPK_0005': 'train',
    'SPK_0006': 'dev', 'SPK_0007': 'test'
}

MANIFEST_COLUMNS = [
    'utt_id', 'audio_path', 'speaker_id', 'label', 'tts_generator',
    'vocoder', 'codec', 'split', 'duration_sec', 'transcript',
    'wer', 'cer', 'ecapa_sim', 'derived_tier', 'engine_name',
    'ref_audio_path', 'seed'
]

def load_and_trim(audio_path, target_sr=16000, top_db=30.0):
    p = Path(audio_path)
    try:
        wav, sr = sf.read(str(p), dtype='float32')
        if wav.ndim > 1:
            wav = np.mean(wav, axis=1)
        if sr != target_sr:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
            sr = target_sr
    except Exception:
        cmd = ['ffmpeg', '-y', '-v', 'error', '-i', str(p), '-ar', str(target_sr), '-ac', '1', '-f', 's16le', '-']
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        wav = np.frombuffer(proc.stdout, dtype=np.int16).astype(np.float32) / 32768.0
        sr = target_sr
    if len(wav) > 512:
        trimmed, _ = librosa.effects.trim(wav, top_db=top_db)
        if len(trimmed) > 0:
            wav = trimmed
    return wav, target_sr

bonafide_records = []
for spk_folder in sorted(raw_src_dir.glob('SPK_*')):
    spk_id = spk_folder.name
    raw_spk = raw_out_dir / spk_id
    ref_spk = ref_out_dir / spk_id
    raw_spk.mkdir(parents=True, exist_ok=True)

    for audio_file in sorted(spk_folder.glob('*')):
        if audio_file.suffix.lower() not in ['.wav', '.m4a', '.mp3', '.flac', '.ogg']:
            continue
        stem = audio_file.stem
        wav, sr = load_and_trim(audio_file)
        dst_raw = raw_spk / f'{stem}.wav'
        sf.write(str(dst_raw), wav, sr, subtype='PCM_16')
        dur = len(wav) / sr

        meta = AUDIO_METADATA.get(stem, {})
        is_ref = meta.get('is_ref', False)
        trans = meta.get('transcript', '')

        if is_ref:
            ref_spk.mkdir(parents=True, exist_ok=True)
            dst_ref = ref_spk / f'{stem}.wav'
            sf.write(str(dst_ref), wav, sr, subtype='PCM_16')

        bonafide_records.append({
            'utt_id': stem,
            'audio_path': f'data/raw/{spk_id}/{stem}.wav',
            'speaker_id': spk_id,
            'label': 'bonafide',
            'tts_generator': 'none',
            'vocoder': 'bonafide',
            'codec': 'clean_16khz',
            'split': SPEAKER_SPLIT.get(spk_id, 'train'),
            'duration_sec': round(dur, 4),
            'transcript': trans,
            'wer': None,
            'cer': None,
            'ecapa_sim': None,
            'derived_tier': 'BonaFide',
            'engine_name': None,
            'ref_audio_path': None,
            'seed': None,
        })

bonafide_df = pd.DataFrame(bonafide_records)[MANIFEST_COLUMNS]
bonafide_csv = manifests_dir / 'bonafide_manifest.csv'
bonafide_df.to_csv(bonafide_csv, index=False, encoding='utf-8')
print(f'✅ Đã chuẩn hóa {len(bonafide_df)} file bonafide -> {bonafide_csv}')


In [ ]:
# 4. Vá lỗi QualityMetrics & Sinh giọng Spoof qua Dual Pipeline F5-TTS / VieNeu-TTS
from vitruevoice import evaluator as _ev_mod
from vitruevoice import pipeline as _pl_mod
from vitruevoice.batch_processor import BatchDatasetProcessor
from vitruevoice.pipeline import ViTrueVoiceDualPipeline, PipelineMode
from vitruevoice.evaluator import MultiGateEvaluator

# --- BƯỚC VÁ LỖI (HOTFIX) CHO QUALITYMETRICS & SYNTHESISRESULT ---
_METRIC_ALIASES = {
    'clean_wer': ('clean_wer', 'wer', 'raw_wer'),
    'wer': ('wer', 'clean_wer', 'raw_wer'),
    'raw_wer': ('raw_wer', 'clean_wer', 'wer'),
    'clean_cer': ('clean_cer', 'cer', 'raw_cer'),
    'cer': ('cer', 'clean_cer', 'raw_cer'),
    'raw_cer': ('raw_cer', 'clean_cer', 'cer'),
    'speaker_sim': ('speaker_sim', 'ecapa_sim', 'sim', 'cosine_sim'),
    'ecapa_sim': ('ecapa_sim', 'speaker_sim', 'sim', 'cosine_sim'),
    'derived_tier': ('derived_tier', 'tier', 'quality_tier'),
    'tier': ('tier', 'derived_tier', 'quality_tier'),
}

def _qm_resolve(self, key):
    d = object.__getattribute__(self, '__dict__')
    if key in d:
        return d[key]
    for alias in _METRIC_ALIASES.get(key, ()):
        if alias in d:
            return d[alias]
    cls = type(self)
    for alias in (key, *_METRIC_ALIASES.get(key, ())):
        if hasattr(cls, alias):
            return object.__getattribute__(self, alias)
    raise KeyError(key)

def _qm_getattr(self, name):
    try:
        return _qm_resolve(self, name)
    except KeyError:
        raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")

if hasattr(_ev_mod, 'QualityMetrics'):
    _QM = _ev_mod.QualityMetrics
    _QM.__getitem__ = _qm_resolve
    _QM.get = lambda self, k, default=None: _qm_resolve(self, k) if k in self else default
    _QM.__contains__ = lambda self, k: any(
        a in object.__getattribute__(self, '__dict__') for a in (k, *_METRIC_ALIASES.get(k, ()))
    )
    _QM.__getattr__ = _qm_getattr

if hasattr(_pl_mod, 'SynthesisResult'):
    _SR = _pl_mod.SynthesisResult
    def _sr_getattr(self, name):
        d = object.__getattribute__(self, '__dict__')
        m = d.get('metrics')
        if m is not None and name in ('speaker_sim', 'ecapa_sim', 'clean_wer', 'clean_cer', 'wer', 'cer'):
            return m.get(name)
        raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")
    _SR.__getattr__ = _sr_getattr

# Vá thêm trong batch_processor.py nếu có check isinstance(metrics, dict)
_bp_file = working_dir / 'vitruevoice' / 'batch_processor.py'
if _bp_file.exists():
    _bp_code = _bp_file.read_text(encoding='utf-8')
    if 'isinstance(metrics, dict)' in _bp_code:
        _bp_file.write_text(_bp_code.replace('isinstance(metrics, dict)', '(metrics is not None)'), encoding='utf-8')

print('✅ Đã áp dụng bản vá cho QualityMetrics & SynthesisResult.')

# --- KHỞI TẠO PIPELINE (TÁI SỬ DỤNG NẾU ĐÃ NẠP TRÊN GPU) ---
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if 'processor' not in globals() or processor is None:
    print(f'🚀 Đang khởi tạo Dual Pipeline và Evaluator trên thiết bị: {device}...')
    evaluator = MultiGateEvaluator(device=device)
    pipeline = ViTrueVoiceDualPipeline(device=device, evaluator=evaluator)
    processor = BatchDatasetProcessor(pipeline=pipeline, evaluator=evaluator, device=device)
else:
    print(f'♻️ Tái sử dụng Pipeline & Evaluator đang có sẵn trên {device} (tránh tràn VRAM).')

spoof_f5_dir = data_root / 'spoof_f5tts'
spoof_vn_dir = data_root / 'spoof_vieneu'
spoof_f5_dir.mkdir(parents=True, exist_ok=True)
spoof_vn_dir.mkdir(parents=True, exist_ok=True)

ref_transcript_map = {k: v.get('transcript', '') for k, v in AUDIO_METADATA.items()}
all_spoof_records = []
for spk_folder in sorted(raw_out_dir.glob('SPK_*')):
    spk_id = spk_folder.name
    wav_files = sorted([f.as_posix() for f in spk_folder.glob('*.wav')])
    if not wav_files:
        continue

    ref_stems = {f.stem for f in (ref_out_dir / spk_id).glob('*.wav')}
    spk_df = bonafide_df[bonafide_df['speaker_id'] == spk_id]
    non_ref_df = spk_df[~spk_df['utt_id'].isin(ref_stems)]
    prompts = non_ref_df[non_ref_df['transcript'].notna()]['transcript'].tolist()

    print(f'\n🎙️ Đang xử lý {spk_id} ({len(prompts)} câu thoại, lọc chất lượng 4 Cổng so khớp mẫu tham chiếu)...')
    records = processor.process_speaker(
        speaker_id=spk_id,
        speaker_wav_files=wav_files,
        prompts=prompts,
        out_synth_dir=data_root,
        mode=PipelineMode.DUAL_OUTPUT,
        apply_telephony=True,
        only_pass=True,
        transcript_map=ref_transcript_map,
        max_retries=2,
        out_f5_dir=spoof_f5_dir / spk_id,
        out_vn_dir=spoof_vn_dir / spk_id
    )

    all_spoof_records.extend(records)
    print(f'✔️ Hoàn tất {spk_id}: sinh được {len(records)} mẫu spoof đạt chuẩn (F5-TTS + VieNeu-TTS x Studio + G.711 + AMR-NB)')

spoof_csv = manifests_dir / 'spoof_manifest.csv'
BatchDatasetProcessor.export_manifest_csv(all_spoof_records, str(spoof_csv))
print(f'\n✅ Đã hoàn tất sinh toàn bộ spoof! Lưu manifest tại: {spoof_csv}')


In [ ]:
# 5. Phân chia tập Speaker-Disjoint & tạo train.csv, dev.csv, test_indomain.csv
spoof_raw_df = pd.read_csv(spoof_csv)

spoof_standard_records = []
for _, r in spoof_raw_df.iterrows():
    p = Path(str(r['audio_path']))
    spk = str(r['speaker_id'])
    eng = str(r.get('engine_name', 'F5-TTS'))
    voc = 'Vocos' if 'f5' in eng.lower() else 'MOSS_48kHz'
    dur = float(r['duration_sec']) if pd.notna(r.get('duration_sec')) else (len(sf.read(str(p))[0]) / 16000 if p.exists() else 4.0)
    rel_path = f'data/{p.parent.parent.name}/{spk}/{p.name}'

    raw_wer_val = r.get('raw_wer', r.get('clean_wer', r.get('wer')))
    raw_cer_val = r.get('raw_cer', r.get('clean_cer', r.get('cer')))
    sim_val = r.get('ecapa_sim', r.get('speaker_sim'))

    spoof_standard_records.append({
        'utt_id': p.stem,
        'audio_path': rel_path,
        'speaker_id': spk,
        'label': 'spoof',
        'tts_generator': eng,
        'vocoder': voc,
        'codec': str(r.get('codec', 'clean_16khz')),
        'split': SPEAKER_SPLIT.get(spk, 'train'),
        'duration_sec': round(dur, 4),
        'transcript': str(r.get('transcript', '')),
        'wer': float(raw_wer_val) if pd.notna(raw_wer_val) else None,
        'cer': float(raw_cer_val) if pd.notna(raw_cer_val) else None,
        'ecapa_sim': float(sim_val) if pd.notna(sim_val) else None,
        'derived_tier': str(r.get('derived_tier', 'Tier 1 (Strong)')),
        'engine_name': eng,
        'ref_audio_path': str(r.get('ref_audio_path', '')),
        'seed': int(r['seed']) if pd.notna(r.get('seed')) else 100,
    })

full_spoof_df = pd.DataFrame(spoof_standard_records)[MANIFEST_COLUMNS]
full_df = pd.concat([bonafide_df, full_spoof_df], ignore_index=True)

train_df = full_df[full_df['split'] == 'train'].copy().reset_index(drop=True)
dev_df = full_df[full_df['split'] == 'dev'].copy().reset_index(drop=True)
test_df = full_df[full_df['split'] == 'test'].copy().reset_index(drop=True)

train_df.to_csv(manifests_dir / 'train.csv', index=False, encoding='utf-8')
dev_df.to_csv(manifests_dir / 'dev.csv', index=False, encoding='utf-8')
test_df.to_csv(manifests_dir / 'test_indomain.csv', index=False, encoding='utf-8')

print('📊 Phân chia tập dữ liệu 17 cột thành công:')
print(f" - Train ({train_df['speaker_id'].nunique()} spks): {len(train_df)} mẫu")
print(f" - Dev   ({dev_df['speaker_id'].nunique()} spks): {len(dev_df)} mẫu")
print(f" - Test  ({test_df['speaker_id'].nunique()} spks): {len(test_df)} mẫu")


In [ ]:
# 6. Đóng gói thư mục data thành file ZIP cho Kaggle Output
import zipfile
import shutil

for m in manifests_dir.glob('*.csv'):
    shutil.copy(m, working_dir / m.name)

zip_out = working_dir / 'vitruevoice_data_processed.zip'
print(f'📦 Đang nén thư mục data vào {zip_out.name}...')
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in data_root.rglob('*'):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(working_dir))

print(f'✅ Đã tạo gói dữ liệu thành công: {zip_out} ({zip_out.stat().st_size / 1e6:.2f} MB)')
print('Tải kết quả về máy local: python manage_kaggle.py output')
